# Advanced RAG Lab – PDF Ingestion with Chroma + Ollama

This notebook demonstrates a **production-style Retrieval Augmented Generation pipeline**.

Features:

- Upload **multiple PDFs**
- Extract and **chunk documents with overlap**
- Store **embeddings + metadata** in Chroma
- Retrieve **top‑k documents with similarity scores**
- Generate answers using **Ollama local LLM**

This lab simulates a **real research workflow for document QA systems**.


In [ ]:
import os
import requests

BASE_URL = os.getenv("OLLAMA_BASE_URL","http://localhost:11434")

print("Using Ollama endpoint:", BASE_URL)

try:
    r = requests.get(BASE_URL,timeout=3)
    print("Ollama reachable:", r.status_code)
except Exception as e:
    print("Connection problem:", e)



## Install dependencies (only if running outside Docker)

In [ ]:
# Uncomment if needed
# !pip install chromadb sentence-transformers pypdf ipywidgets


## Imports

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
import ipywidgets as widgets
from IPython.display import display
import uuid



## Initialize Vector Database

In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="rag_documents"
)

print("Chroma collection ready")



## Load Embedding Model

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded")



## Upload one or more PDFs

In [ ]:
uploader = widgets.FileUpload(
    accept='.pdf',
    multiple=True
)

display(uploader)



## Extract text from PDFs

In [ ]:
documents = []

for uploaded_file in uploader.value:

    name = uploaded_file["name"]
    content = uploaded_file["content"]

    # salva il pdf su disco
    with open(name, "wb") as f:
        f.write(content)

    reader = PdfReader(name)

    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    documents.append({
        "name": name,
        "text": text
    })

print("Documents processed:", len(documents))



## Chunk documents with overlap

In [ ]:
chunk_size = 500
overlap = 100

chunks = []

for doc in documents:

    text = doc["text"]
    source = doc["name"]

    for i in range(0, len(text), chunk_size - overlap):

        chunk = text[i:i+chunk_size]

        chunks.append({
            "text": chunk,
            "source": source
        })

print("Total chunks:", len(chunks))



## Generate embeddings and store in Chroma

In [ ]:
texts = [c["text"] for c in chunks]

embeddings = model.encode(texts)

ids = [str(uuid.uuid4()) for _ in texts]

metadatas = [{"source":c["source"]} for c in chunks]

collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=ids,
    metadatas=metadatas
)

print("Chunks indexed in vector DB")



## Retrieval function with scores

In [ ]:
def retrieve(query, k=4):

    q_emb = model.encode([query])[0]

    results = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]

    hits = []

    for d,m,s in zip(docs,metas,distances):
        hits.append({
            "text": d,
            "source": m["source"],
            "score": s
        })

    return hits



## Test retrieval

In [ ]:
hits = retrieve("What topics are discussed in the document?", k=3)

for i,h in enumerate(hits):

    print("---- Result",i+1,"----")
    print("Source:",h["source"])
    print("Score:",h["score"])
    print(h["text"][:400])
    print()



## Ollama generation

In [ ]:
def ollama_generate(prompt,model="llama3.2:3b"):

    payload = {
        "model":model,
        "prompt":prompt,
        "stream":False
    }

    r = requests.post(
        f"{BASE_URL}/api/generate",
        json=payload
    )

    r.raise_for_status()

    return r.json()["response"]



## Full RAG pipeline

In [ ]:
def rag_answer(question, k=4):

    hits = retrieve(question,k)

    context = "\n\n".join(
        [f"[{i+1}] ({h['source']}) {h['text']}" for i,h in enumerate(hits)]
    )

    print("Retrieved context:\n")
    print(context[:1200])
    print("\n---\n")

    prompt = f"""You are a helpful assistant.

Use ONLY the context below to answer the question.

If the answer cannot be found in the context say:
'I don't know based on the provided documents.'

Context:
{context}

Question: {question}

Answer:"""

    return ollama_generate(prompt)



## Ask a question about the PDFs

In [ ]:
answer = rag_answer(
    "Summarize the main ideas of the uploaded documents."
)

print(answer)

